# FinBERT Fine-Tuning on financial_phrasebank
**Run this on Kaggle with GPU accelerator enabled (T4 x2 or P100)**

Self-contained — no project files needed. Output saved to `/kaggle/working/finbert-finetuned/`

In [ ]:
# Cell 1 — Install / upgrade packages
# Pin datasets to 2.x — v3.0 dropped support for custom dataset scripts
# (financial_phrasebank still uses one). Everything else can be latest.
!pip install -q \
    'transformers>=4.39.0' \
    'datasets>=2.18.0,<3.0.0' \
    'scikit-learn>=1.4.0' \
    accelerate

# Confirm the pinned version is active in this session
import importlib, datasets
importlib.reload(datasets)
print('datasets version:', datasets.__version__)   # must be 2.x

In [ ]:
# Cell 2 — Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Cell 3 — Config (edit if needed)
BASE_MODEL   = 'ProsusAI/finbert'
OUTPUT_DIR   = '/kaggle/working/finbert-finetuned'
EPOCHS       = 3
BATCH_SIZE   = 32    # T4 can handle 32; drop to 16 if you see OOM
LR           = 2e-5
WARMUP_RATIO = 0.1
MAX_LENGTH   = 512
SEED         = 42

# financial_phrasebank label encoding (from the dataset)
# 0 = negative, 1 = neutral, 2 = positive
ID2LABEL = {0: 'negative', 1: 'neutral', 2: 'positive'}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

In [ ]:
# Cell 4 — Load dataset
# financial_phrasebank uses a custom loading script, which requires datasets<3.0.
# If the pip pin above worked you'll see "Generating train split..." here.
# Fallback: we build the Dataset from the raw HuggingFace parquet export directly.

import pandas as pd
from datasets import load_dataset, Dataset

def _load_via_parquet() -> Dataset:
    """
    Fallback loader: pulls the pre-built parquet file that HuggingFace
    generates for every dataset, regardless of whether it uses a script.
    Works with datasets 3.x.
    """
    print('Using parquet fallback loader...')
    # HuggingFace exposes a parquet conversion for every dataset via the API
    url = (
        'https://huggingface.co/datasets/takala/financial_phrasebank'
        '/resolve/main/data/sentences_allagree-train.parquet'
    )
    try:
        df = pd.read_parquet(url)
    except Exception:
        # Second fallback: the datasets-server REST endpoint
        import requests, io
        resp = requests.get(
            'https://datasets-server.huggingface.co/rows'
            '?dataset=takala%2Ffinancial_phrasebank'
            '&config=sentences_allagree&split=train&offset=0&limit=5000',
            timeout=60,
        )
        resp.raise_for_status()
        rows = [r['row'] for r in resp.json()['rows']]
        df = pd.DataFrame(rows)

    # Normalise column names to what the rest of the notebook expects
    if 'Sentence' in df.columns:
        df = df.rename(columns={'Sentence': 'sentence', 'Sentiment': 'label'})
    label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
    if df['label'].dtype == object:
        df['label'] = df['label'].map(label_map)
    return Dataset.from_pandas(df[['sentence', 'label']].reset_index(drop=True))


# --- Primary load (works with datasets 2.x) ---
try:
    raw = load_dataset('takala/financial_phrasebank', 'sentences_allagree', trust_remote_code=True)
    full_ds = raw['train']
    print('Loaded via load_dataset.')
except RuntimeError as e:
    if 'scripts are no longer supported' in str(e):
        full_ds = _load_via_parquet()
        print('Loaded via parquet fallback.')
    else:
        raise

split    = full_ds.train_test_split(test_size=0.15, seed=SEED, stratify_by_column='label')
train_ds = split['train']
eval_ds  = split['test']

print(f'Train: {len(train_ds):,} | Eval: {len(eval_ds):,}')
from collections import Counter
counts = Counter(train_ds['label'])
print('Label distribution (train):', {ID2LABEL[k]: v for k, v in sorted(counts.items())})

In [ ]:
# Cell 5 — Tokenise
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    return tokenizer(
        batch['sentence'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_ds = train_ds.map(tokenize, batched=True, remove_columns=['sentence'])
eval_ds  = eval_ds.map(tokenize,  batched=True, remove_columns=['sentence'])
train_ds.set_format('torch')
eval_ds.set_format('torch')
print('Tokenisation done.')

In [ ]:
# Cell 6 — Load model
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,   # replaces the 3-class head from ProsusAI
)
print('Model loaded. Parameters:', sum(p.numel() for p in model.parameters()) // 1_000_000, 'M')

In [ ]:
# Cell 7 — Metrics
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, classification_report

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':    accuracy_score(labels, preds),
        'weighted_f1': f1_score(labels, preds, average='weighted'),
        'macro_f1':    f1_score(labels, preds, average='macro'),
    }

In [ ]:
# Cell 8 — Train
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='weighted_f1',
    greater_is_better=True,
    logging_steps=20,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),   # free speed on CUDA; safe on T4/P100
    report_to='none',
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

In [ ]:
# Cell 9 — Evaluate & print report
eval_results = trainer.evaluate()
print('\n=== Eval results ===')
for k, v in eval_results.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

preds = trainer.predict(eval_ds)
pred_labels = np.argmax(preds.predictions, axis=-1)
true_labels = eval_ds['label'].numpy()
print('\n=== Classification Report ===')
print(classification_report(true_labels, pred_labels, target_names=list(ID2LABEL.values())))

In [ ]:
# Cell 10 — Save final model
import os
final_dir = os.path.join(OUTPUT_DIR, 'final')
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)

# Write a small metadata file so you know what produced this artifact
import json
meta = {
    'base_model': BASE_MODEL,
    'dataset': 'takala/financial_phrasebank (sentences_allagree)',
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LR,
    'weighted_f1': round(eval_results['eval_weighted_f1'], 4),
    'accuracy': round(eval_results['eval_accuracy'], 4),
    'id2label': ID2LABEL,
}
with open(os.path.join(final_dir, 'training_meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print(f'\nModel saved to {final_dir}')
print('Files:', os.listdir(final_dir))

In [ ]:
# Cell 11 — Zip for download
import shutil
zip_path = '/kaggle/working/finbert-finetuned-final'
shutil.make_archive(zip_path, 'zip', final_dir)
print(f'Download ready: {zip_path}.zip')
print(f'Size: {os.path.getsize(zip_path + ".zip") / 1e6:.1f} MB')